# NóiLái — API runs (Gemini and Groq free tiers)

Runs the API lines of `configs/run_plan.yaml` (`E1_api_core`, `E1_api_paraphrase`,
`E3_api_core`, `reasoning_500_api`) on the API-served models. Guards in
`scripts/kaggle_run_plan.py` make every API command `--in-core-only`, refuse the sealed split,
and refuse every canary-bearing (test-split) item file for a provider whose tier trains on
inputs (DD 11.2 / 12.41, item 33): **Gemini's unpaid tier never receives the core.** As shipped
the Gemini models are therefore reported as *refused* on every core-derived line and only the
Groq models run; Gemini runs once the author has either recorded a paid key with a verified
data-use opt-out (`providers.gemini.terms.paid_key_data_use_opt_out: true` in `configs/models.yaml`)
or added the dev-derived API set `noilai_api_dev` (DD 4.5) with its own lines, reported
separately and never in Table 2 (DD 13.18). Keys come from the environment / Colab userdata /
Kaggle Secrets (`GEMINI_API_KEY`, `GROQ_API_KEY`) and are never printed.

**Daily-quota loop.** Groq's free plan allows 1,000 requests and 200K tokens a day; at
~500 tokens a call the TOKEN cap binds first (~400 calls), so the core takes about four days
per model. The driver keeps `data/runs/api_ledger.json` (requests AND tokens per model per
UTC day, counted from the new rows of `outputs.jsonl`), skips a model whose daily request or
token budget is spent, and stops a running command (SIGINT, rows kept) the moment the day's
cap is reached — reported as *parked*, not failed; the backend paces itself at the entry's
`requests_per_minute`. Run this notebook once a day until every run is complete: the clone is
ephemeral, so cell (b') first restores the previous day's `data/runs` (outputs, manifests,
`api_ledger.json`) from `RUNS_RESTORE_DIR` — the Drive folder the last cell copied to — and
`--resume` then continues at the first unanswered row with yesterday's ledger, instead of
re-requesting the same rows against a fresh daily allowance. Gemini's free-tier limits are shown
only in AI Studio: read them and set `GEMINI_RPD_OVERRIDE` / `GEMINI_TPD_OVERRIDE` (DD 7.2); a
provider with no configured cap and no override is refused unless `ALLOW_UNCAPPED_API`; the
override is applied to the in-memory config and recorded in the ledger — no file under
`configs/` is ever rewritten by a notebook.

Terms: Gemini's free tier trains on inputs and requires users to be 18+; the core reaches only
providers with no-training terms (Groq); never annotator data, never the sealed split (DD 11.2).
Set `ACCOUNT_HOLDER_ROLE` (a role, never a name) so that every manifest records who held the
API accounts (DD 11.2).

In [ ]:
# ---- parameters: edit this cell only; nothing below asks for input ------------------------------
import time
SESSION_T0 = time.time()

REPO_URL = "https://github.com/BEEEEEEBAGON/noilai.git"   # the split-out repository (docs/MIGRATION.md); BUNDLE_PATH wins when set
REPO_REF = "main"
REPO_SUBDIR = "noilai"
BUNDLE_PATH = "/content/drive/MyDrive/noilai/noilai-main.bundle"   # Colab; on Kaggle use /kaggle/input/noilai-bundle/noilai-main.bundle
CLONE_DIR = "/content/repo"
WORK_DIR = "/content"
ITEMS_DATASET_DIR = "/content/drive/MyDrive/noilai/release"        # the frozen data/release tree; None if the clone has it
RUNS_RESTORE_DIR = "/content/drive/MyDrive/noilai/runs_api"        # where the last cell copied data/runs (+ the ledger) to; restored by (b') so --resume and the ledger continue; on Kaggle: /kaggle/input/noilai-runs
VERIFY_ITEM_KEYS = None          # None = derived from RUN_IDS by (d)

GITHUB_TOKEN_SECRET = "GITHUB_TOKEN"
API_KEY_SECRETS = ["GEMINI_API_KEY", "GROQ_API_KEY"]   # names only; values stay in the secret store
GEMINI_RPD_OVERRIDE = None       # requests/day read in AI Studio; None = no ledger cap for Gemini [UNCERTAIN: verify]; applied in memory, recorded in the ledger
GEMINI_TPD_OVERRIDE = None       # tokens/day read in AI Studio, same handling [UNCERTAIN: verify]
ALLOW_UNCAPPED_API = False       # DD 7.2 / 7.3: a selected provider with neither a configured daily cap nor an override is refused unless True (recorded in the ledger)

RUN_IDS = ["E1_api_core", "E1_api_paraphrase", "E3_api_core"]   # add "reasoning_500_api" for the cost curve
MODELS = []                      # subset of the run's models; [] = every model of the line (the Gemini ones are refused on core-derived files until DD 13.18 is decided)
EXTRA_ARGS = ""
ACCOUNT_HOLDER_ROLE = ""         # ROLE of the API account holder (e.g. "author", "adult collaborator"); never a name (DD 11.2); recorded in every manifest
DRY_RUN = False

PLATFORM, GPU_TYPE, N_GPUS = "api", "api", 0
SESSION_HARDWARE = "api"
MAX_SESSION_HOURS = None         # the API loop is bounded by the daily caps, not by a session
REQUIRE_GPU = False
RUN_LABEL = "+".join(RUN_IDS)
MOUNT_DRIVE = True

In [ ]:
# secrets: environment -> Colab userdata -> Kaggle Secrets. Values go into os.environ for the
# library that needs them and are reported as found / not found. Nothing here prints a value.
import os

def _secret(name):
    val = os.environ.get(name)
    if val:
        return val
    try:
        from google.colab import userdata          # Colab
        try:
            val = userdata.get(name)
        except Exception:
            val = None
        if val:
            return val
    except ImportError:
        pass
    try:
        from kaggle_secrets import UserSecretsClient   # Kaggle
        try:
            val = UserSecretsClient().get_secret(name)
        except Exception:
            val = None
        if val:
            return val
    except ImportError:
        pass
    return None

def _export(name, required=False):
    val = _secret(name)
    if val:
        os.environ[name] = val
        print(f"secret {name}: found ({len(val)} chars, not shown)")
        return True
    print(f"secret {name}: not found")
    if required:
        raise SystemExit(f"{name} is required for this notebook")
    return False

In [ ]:
# Drive (Colab only): bundle, release and the outputs' backup
from pathlib import Path
if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        print("Drive mounted")
    except ImportError:
        print("not on Colab: no Drive; set BUNDLE_PATH / ITEMS_DATASET_DIR to local paths")

In [ ]:
# (b) clone: from a git bundle in a private dataset / Drive when present (no token needed), else from
# GitHub with a token from the platform's secret store. The token reaches git only through GIT_ASKPASS,
# a two-line helper that echoes an environment variable: it is never in a URL, an argv or this output.
import os, stat, subprocess, sys
from pathlib import Path

clone_dir = Path(CLONE_DIR)
bundle = Path(BUNDLE_PATH) if BUNDLE_PATH else None
if bundle is not None and not bundle.exists():
    print(f"bundle {bundle} not found; falling back to {REPO_URL}")
    bundle = None
askpass = None
if bundle is None:
    if "OWNER/REPO" in REPO_URL:
        raise SystemExit("set REPO_URL (or provide BUNDLE_PATH) in the parameters cell")
    if _export(GITHUB_TOKEN_SECRET):
        askpass = Path.home() / ".noilai_askpass.sh"
        askpass.write_text("#!/bin/sh\necho \"$" + GITHUB_TOKEN_SECRET + "\"\n")
        askpass.chmod(stat.S_IRWXU)
# the environment git runs in is built AFTER the export, so the askpass helper finds the token in it
env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
if askpass is not None:
    env["GIT_ASKPASS"] = str(askpass)
source = str(bundle) if bundle else REPO_URL
if (clone_dir / ".git").exists():
    subprocess.run(["git", "-C", str(clone_dir), "fetch", "--tags", source, "+refs/heads/*:refs/remotes/origin/*"], check=True, env=env)
else:
    subprocess.run(["git", "clone", source, str(clone_dir)], check=True, env=env)
# a persisted clone may hold a STALE local branch named like REPO_REF: try the freshly fetched
# remote-tracking ref first and the literal ref last (commit hashes and tags have no remote form)
import re
cands = [REPO_REF] if re.fullmatch(r"[0-9a-f]{7,40}", REPO_REF) else [f"origin/{REPO_REF}", REPO_REF]
for cand in cands:
    if subprocess.run(["git", "-C", str(clone_dir), "checkout", "--detach", cand], env=env).returncode == 0:
        break
else:
    raise SystemExit(f"cannot check out {REPO_REF} (tried {cands})")
HEAD = subprocess.run(["git", "-C", str(clone_dir), "rev-parse", "HEAD"], check=True, env=env,
                      capture_output=True, text=True).stdout.strip()
PROJECT = clone_dir / REPO_SUBDIR if (clone_dir / REPO_SUBDIR / "pyproject.toml").exists() else clone_dir
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT / "scripts"))
print("HEAD", HEAD)
print("project", PROJECT)

In [ ]:
# (b') restore: the clone is ephemeral, so the runs tree persisted by the previous session (the private
# Kaggle dataset `noilai-runs` attached as input, or the Drive folder the API notebook copies to) is
# copied back into PROJECT/data/runs BEFORE anything runs: outputs.jsonl / manifest.json / unchanged.jsonl
# per run directory (so `--resume` continues at the first undone row instead of item 0), api_ledger.json
# (so a day's quota is not re-spent on rows already answered; DD 7.3 "a hit never re-calls") and the
# compute log (so the session's rows append to the campaign's). Nothing here overwrites a newer local file.
import shutil
from pathlib import Path
RESTORED = {"from": None, "run_dirs": 0, "ledger": False, "compute_log": False}
_src = Path(RUNS_RESTORE_DIR) if RUNS_RESTORE_DIR else None
if _src is not None and _src.exists():
    runs_src = _src / "runs" if (_src / "runs").is_dir() else _src        # the dataset holds runs/ + compute_log.csv, or the runs tree itself
    runs_dst = PROJECT / "data" / "runs"
    runs_dst.mkdir(parents=True, exist_ok=True)
    for child in sorted(runs_src.iterdir()):
        if child.name in ("__pycache__", ".ipynb_checkpoints"):
            continue
        if child.is_dir():
            shutil.copytree(child, runs_dst / child.name, dirs_exist_ok=True)
            RESTORED["run_dirs"] += 1
        elif child.name == "api_ledger.json":
            shutil.copy2(child, runs_dst / child.name)
            RESTORED["ledger"] = True
        elif child.suffix in (".json", ".jsonl", ".txt", ".csv"):
            shutil.copy2(child, runs_dst / child.name)
    for cand in (_src / "compute_log.csv", runs_src / "compute_log.csv"):
        if cand.exists() and not (PROJECT / "data" / "compute_log.csv").exists():
            shutil.copy2(cand, PROJECT / "data" / "compute_log.csv")
            RESTORED["compute_log"] = True
            break
    RESTORED["from"] = str(runs_src)
    print("restored", RESTORED)
else:
    print("nothing to restore (first session, or RUNS_RESTORE_DIR", RUNS_RESTORE_DIR, "is not attached): every run starts at item 0")

In [ ]:
# (c) install: the project with its eval extra (API SDKs: openai, google-genai, groq); no inference engine.
# Record pip freeze + versions next to the outputs (checklist C4).
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[eval]"], check=True)
RUN_PYTHON = sys.executable
import colab_setup as CS
ENV_RECORD = CS.record_environment(PROJECT / "data" / "runs" / "env")
print("environment recorded at", ENV_RECORD)

In [ ]:
# API keys into the environment (names from configs/models.yaml providers.*.api_key_env); found / not found only.
# Session overrides of the Gemini caps are applied to the IN-MEMORY config the driver receives (MODELS_CFG),
# never written to configs/models.yaml (that would strip its comments and dirty every manifest's git state).
import kaggle_run_plan as KRP
PLAN, MODELS_CFG = KRP.load_plan(), KRP.load_models()
wanted = sorted({p["api_key_env"] for p in MODELS_CFG["providers"].values()} & set(API_KEY_SECRETS))
FOUND = {name: _export(name) for name in wanted}
if not any(FOUND.values()):
    raise SystemExit("no API key found: add GEMINI_API_KEY / GROQ_API_KEY to the secret store")
OVERRIDES = {}
if GEMINI_RPD_OVERRIDE:
    MODELS_CFG["providers"]["gemini"]["free_tier"]["requests_per_day"] = int(GEMINI_RPD_OVERRIDE)
    OVERRIDES["gemini.requests_per_day"] = int(GEMINI_RPD_OVERRIDE)
if GEMINI_TPD_OVERRIDE:
    MODELS_CFG["providers"]["gemini"]["free_tier"]["tokens_per_day"] = int(GEMINI_TPD_OVERRIDE)
    OVERRIDES["gemini.tokens_per_day"] = int(GEMINI_TPD_OVERRIDE)
# DD 7.2 / 7.3: the ledger can only park a model at a cap it knows. Every provider of the selected models
# must have a daily request or token cap (configured or overridden) unless ALLOW_UNCAPPED_API says so.
_selected = set(MODELS) or {n for rid in RUN_IDS for n in KRP.expand_models(KRP.find_run(PLAN, rid), PLAN, MODELS_CFG)}
_uncapped = sorted({MODELS_CFG["by_name"][n]["provider"] for n in _selected
                    if MODELS_CFG["by_name"][n].get("provider") and os.environ.get(MODELS_CFG["providers"][MODELS_CFG["by_name"][n]["provider"]]["api_key_env"])
                    and not (MODELS_CFG["providers"][MODELS_CFG["by_name"][n]["provider"]]["free_tier"].get("requests_per_day")
                             or MODELS_CFG["providers"][MODELS_CFG["by_name"][n]["provider"]]["free_tier"].get("tokens_per_day"))})
OVERRIDES["allow_uncapped_api"] = bool(ALLOW_UNCAPPED_API)
if _uncapped and not ALLOW_UNCAPPED_API:
    raise SystemExit(f"provider(s) {_uncapped} have no daily cap in configs/models.yaml and no override: read the live quota in "
                     "AI Studio and set GEMINI_RPD_OVERRIDE / GEMINI_TPD_OVERRIDE (DD 7.2), or set ALLOW_UNCAPPED_API = True")
lpath = KRP.ledger_path(PLAN)
ledger = KRP.ledger_load(lpath)
ledger.setdefault("_overrides", {})[KRP.utc_day()] = OVERRIDES
KRP.ledger_save(lpath, ledger)
print("session overrides (in memory, recorded in the ledger):", OVERRIDES)

In [ ]:
# (d) third-party resources (SHA-256 against data/HASHES.json) and the item file(s) of the runs this
# notebook will execute: the keys are DERIVED from the run ids (plus the smoke run) unless
# VERIFY_ITEM_KEYS lists them; a seeded sub-sample the plan derives from another file is written first
# (--materialize). Each file is checked for its hash against configs/run_plan.yaml, the header record,
# the canary on every row and its row/cell counts. A failure stops the notebook here.
import shutil, subprocess, sys
from pathlib import Path
import kaggle_run_plan as KRP
r = subprocess.run([sys.executable, "scripts/fetch_resources.py"], check=False)
if r.returncode:
    raise SystemExit("resource fetch or hash check failed (see above)")
if ITEMS_DATASET_DIR:
    src = Path(ITEMS_DATASET_DIR)
    if not src.exists():
        raise SystemExit(f"ITEMS_DATASET_DIR {src} not found: attach the private dataset that holds data/release")
    rel = Path(KRP.load_plan()["release"])            # e.g. data/release/v0.3: the plan's `release` key decides where the files go
    dest = PROJECT / rel
    if (src / rel.name).exists():                      # the dataset holds the data/release tree (one directory per version)
        shutil.copytree(src / rel.name, dest, dirs_exist_ok=True)
    elif (src / "manifest.json").exists():             # the dataset holds ONE release directory
        shutil.copytree(src, dest, dirs_exist_ok=True)
    else:
        raise SystemExit(f"{src} holds neither {rel.name}/ nor a release manifest.json: attach the frozen release")
    print("release copied from", src, "to", dest)
_run_ids = list(globals().get("RUN_IDS") or [globals().get("RUN_ID")]) + list(globals().get("SMOKE_RUN_IDS") or [])
KEYS = VERIFY_ITEM_KEYS or KRP.item_keys_for_runs(KRP.load_plan(), [r for r in _run_ids if r])
print("item files to verify:", KEYS)
for key in KEYS:
    r = subprocess.run([sys.executable, "scripts/kaggle_verify_items.py", "--key", key, "--materialize"], check=False)
    if r.returncode:
        raise SystemExit(f"item file {key} failed verification; nothing runs on it")

In [ ]:
# after each RUN cell: scripts/check_run.py on every finished run directory: item gate, deterministic rescoring,
# stats.json (accuracy with the base-pair cluster-bootstrap CI, the copy-baseline gain of PREREG section 10, the paired
# NFD contrast, the EXPLORATORY T1 forced choice against chance) and results_hashes.json
import json, subprocess, sys

def check_runs(results):
    summaries = []
    for r in results:
        run_dir = PROJECT / "data" / "runs" / f"{r['run']}__{r['model']}"
        if not (run_dir / "scores.jsonl").exists():
            print(r["run"], r["model"], "->", r["status"], "(no scores: not checked)")
            continue
        c = subprocess.run([sys.executable, "scripts/check_run.py", "--run", str(run_dir)], capture_output=True, text=True, check=False)
        print(c.stdout.strip() or c.stderr[-1500:])
        st = json.loads((run_dir / "stats.json").read_text()) if (run_dir / "stats.json").exists() else {}
        summaries.append({"run": r["run"], "model": r["model"], "status": r["status"],
                          "by_task_arm": {k: (round(v["accuracy"], 3), [round(x, 3) for x in v["ci"]]) for k, v in st.get("by_task_arm", {}).items()},
                          "t1_forced_choice": {k: (round(v["accuracy"], 3), round(v["chance"], 3)) for k, v in st.get("t1_forced_choice", {}).items()}})
    print(json.dumps(summaries, ensure_ascii=False, indent=1))
    return summaries

### RUN CELL — daily API loop
For each run id and model: skip if today's ledger says the daily request OR token budget is
spent, else `run_eval.py ... --in-core-only --resume` under the budget watchdog (the command
is interrupted at the cap and reported as *parked*). Models whose key is missing are skipped
with a message; a training-tier provider on a test-split file is reported as *refused* and the
loop goes on. Re-run tomorrow: (b') restores today's outputs and ledger, `--resume` continues.

In [ ]:
# RUN CELL: quota-aware loop over RUN_IDS x MODELS through scripts/kaggle_run_plan.py (in-core-only enforced there);
# the in-memory MODELS_CFG carries the session overrides.
import json, shlex
import kaggle_run_plan as KRP
RUN_EXTRA = shlex.split(EXTRA_ARGS) + (["--account-holder", ACCOUNT_HOLDER_ROLE] if ACCOUNT_HOLDER_ROLE else [])   # role only, never a name (DD 11.2)
if not ACCOUNT_HOLDER_ROLE:
    print("WARNING: ACCOUNT_HOLDER_ROLE is empty; every API manifest will record account_holder 'unknown' (DD 11.2 asks for the role)")
RESULTS = []
for run_id in RUN_IDS:
    run = KRP.find_run(PLAN, run_id)
    names = MODELS or KRP.expand_models(run, PLAN, MODELS_CFG)
    runnable = []
    for name in names:
        prov = MODELS_CFG["by_name"][name].get("provider")
        key_env = MODELS_CFG["providers"][prov]["api_key_env"] if prov else None
        if key_env and not os.environ.get(key_env):
            print(f"skip {name}: {key_env} not set")
        else:
            runnable.append(name)
    RESULTS += KRP.execute(run_id, models=runnable, platform=PLATFORM, dry_run=DRY_RUN, continue_on_error=True,
                           extra=RUN_EXTRA, plan=PLAN, models_cfg=MODELS_CFG, python=RUN_PYTHON,
                           session_hardware=SESSION_HARDWARE)
print(json.dumps([{"run": r["run"], "model": r["model"], "status": r["status"], "new_outputs": r.get("new_outputs"),
                   "new_tokens": r.get("new_tokens"), "budget": r.get("api_budget_today")} for r in RESULTS], indent=1))
ledger = KRP.ledger_load(KRP.ledger_path(PLAN))
print("ledger (requests and tokens per model per UTC day):", json.dumps(ledger, indent=1))

In [ ]:
# (f') hashed results: scripts/check_run.py on every run directory this session wrote (scores it first if needed; item gate,
# deterministic rescoring, stats.json, results_hashes.json); the copies pushed in (g) carry them
CHECKED = check_runs(list(globals().get("SMOKE_RESULTS", [])) + list(globals().get("RESULTS", [])))

In [ ]:
# keep the outputs: copy data/runs, the ledger and the compute log to Drive (Colab) or the working dir; the
# Drive path is RUNS_RESTORE_DIR, which (b') reads back tomorrow (runs/ + compute_log.csv, the layout (b') expects)
import shutil
from pathlib import Path
dest = Path(RUNS_RESTORE_DIR) if (RUNS_RESTORE_DIR and Path(RUNS_RESTORE_DIR).parent.exists() and Path("/content/drive/MyDrive").exists()) else Path(WORK_DIR) / "noilai_runs_out"
shutil.copytree(PROJECT / "data" / "runs", dest / "runs", dirs_exist_ok=True)
import compute_log as CL
LOG = PROJECT / "data" / "compute_log.csv"
CL.append_entry(LOG, CL.Entry(date=CL.today(), platform=PLATFORM, gpu_type=GPU_TYPE, n_gpus=N_GPUS,
                              hours=CL.hours_since(SESSION_T0), run_id=f"session-{RUN_LABEL}", purpose="api_runs"))
shutil.copy2(LOG, dest / "compute_log.csv")
print(CL.format_totals(CL.totals(CL.read_entries(LOG))))
print("outputs copied to", dest)